# 01 — Data Audit and Cleaning

**Purpose.** Load the supplied KOL snapshot without modifying it, document every data-quality
issue, parse the views field defensively, and produce a processed table that keeps the raw
values beside the derived ones.

**Scope note.** The file is a *cross-sectional snapshot*, not a historical point-in-time
panel. It has no post IDs, no timestamps, no engagement history and no prices. Nothing in
this notebook makes any performance claim.

In [1]:
import sys, json
from pathlib import Path

# Make the repository root importable when the notebook is executed by nbconvert.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src import data as D
from src import plotting as P

P.setup_style()
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 90)
print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.3.3 | numpy 2.2.6


## 1. Load the raw file

`load_raw` resolves Chinese and English column aliases, strips whitespace and BOM characters
from headers, and raises a helpful error if a required field is missing. It reads with
`dtype=str` so that nothing is coerced silently at load time.

In [2]:
RAW_PATH = ROOT / "data" / "raw" / "users.csv"
raw = D.load_raw(RAW_PATH)

print("rows, columns:", raw.shape)
print("canonical columns:", list(raw.columns))
display(raw.head(3))

rows, columns: (119, 4)
canonical columns: ['nickname', 'profile_url', 'views_raw', 'text_raw']


,nickname,profile_url,views_raw,text_raw
0,Bill The Investor,https://x.com/billtheinvestor,30000,AI中转站黑幕曝光：真钱买假模型已成行业潜规则！ 很多低价Claude/GPT-5“满血”通道，其实偷偷换成国产弱模型/旧版3.5/甚至随机路由，赚差价跑路。论文都证实4...
1,墓碑科技,https://x.com/mubeitech,65000,AI教父Yann LeCun说了一个事实。 一个四岁的小孩，看过的数据量，等于有史以来所有文本的总和。 最大的AI模型，训练数据是30万亿个单词。 听着很多？ 一个四岁小...
2,极客杰尼,https://x.com/seekjourney,54000,AI 用得越来越多以后，我发现了两个不一样的世界。 普通玩家：Prompt → Output（输入指令 → 得到结果） 高阶玩家：Context → Alignment...


### Record the file fingerprint

The original `users.csv` at the repository root is never written to. This hash is recorded so
that a reviewer can confirm the same input was used.

In [3]:
import hashlib
digest = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
print("raw file sha256:", digest)
print("bytes:", RAW_PATH.stat().st_size)

raw file sha256: 8d8d63aa2c936aeda18e033d92f6b9569fb637f2823697df3d891c5e0e1c683c
bytes: 35681


## 2. Structural audit

Missingness, uniqueness and duplicates, computed on the raw canonical fields before any
derivation.

In [4]:
audit_missing = D.audit_missing_values(raw)
display(audit_missing)

print("empty strings per field:")
for c in ["nickname", "profile_url", "views_raw", "text_raw"]:
    n_empty = int((raw[c].astype(str).str.strip() == "").sum())
    print(f"  {c:14s} {n_empty}")

print("\nexact full-row duplicates:", int(raw.duplicated().sum()))
print("unique profile_url        :", raw["profile_url"].nunique(), "of", len(raw))
print("unique nickname           :", raw["nickname"].nunique(), "of", len(raw))

,column,dtype,n_missing,pct_missing,n_unique
0,nickname,string,0,0.0,118
1,profile_url,string,0,0.0,119
2,views_raw,string,0,0.0,93
3,text_raw,string,0,0.0,119


empty strings per field:
  nickname       0
  profile_url    0
  views_raw      0
  text_raw       0

exact full-row duplicates: 0
unique profile_url        : 119 of 119
unique nickname           : 118 of 119


### Duplicate nickname, distinct accounts

One nickname appears twice with two different handles. This is *not* a duplicate record; it is
the same person with a second account. We keep both and flag the nickname collision so that
any author-level aggregation is aware of it.

In [5]:
dupe_nicks = raw.loc[raw["nickname"].duplicated(keep=False), ["nickname", "profile_url"]]
display(dupe_nicks.sort_values("nickname"))

,nickname,profile_url
16,yetone,https://x.com/yetone
104,yetone,https://x.com/7id


### Username extraction

The handle is derived from the profile URL rather than parsed from free text, and validated
against the `x.com` pattern. Any URL that does not match is reported instead of dropped.

In [6]:
clean0 = D.build_clean_frame(raw)
bad_url = clean0[~clean0["profile_url"].str.match(r"^https://x\.com/[A-Za-z0-9_]+$")]
print("profile URLs failing the x.com pattern:", len(bad_url))
if len(bad_url):
    display(bad_url[["row_id", "profile_url"]])

display(clean0[["row_id", "nickname", "username", "views_raw", "views_numeric",
                "views_parse_status", "log_views"]].head(8))

profile URLs failing the x.com pattern: 0


,row_id,nickname,username,views_raw,views_numeric,views_parse_status,log_views
0,0,Bill The Investor,billtheinvestor,30000,30000.0,parsed,10.308986
1,1,墓碑科技,mubeitech,65000,65000.0,parsed,11.082158
2,2,极客杰尼,seekjourney,54000,54000.0,parsed,10.896758
3,3,Zaotang,zaotang18,48000,48000.0,parsed,10.778977
4,4,Ken W,kenw_2,14000,14000.0,parsed,9.546884
5,5,李岳,liyue_ai,240000,240000.0,parsed,12.388398
6,6,Mr Panda,PandaTalk8,160000,160000.0,parsed,11.982935
7,7,Geek Lite,QingQ77,3350,3350.0,parsed,8.117014


## 3. Views: robust parsing

The supplied `阅读数` column is already an integer. The parser is nevertheless written to
handle `"1,234"`, `"12K"`, `"1.5M"`, `"3.4万"`, `"1.2亿"`, unit suffixes and blanks, because the
production version of this export will not always be clean.

**Key rule:** an unparseable value becomes `NaN`, never `0`. A genuine zero is preserved and
labelled `explicit_zero` so the two cases can never be confused.

In [7]:
parser_cases = ["1,234", "12K", "1.5M", "3.4万", "1.2亿", "3350", "  19,000 ",
                "12k次", "", "  ", "abc", "K", "0", "-5", None]
cases_df = pd.DataFrame({
    "input": [repr(v) for v in parser_cases],
    "parsed": [D.parse_views(v) for v in parser_cases],
})
display(cases_df)

status_counts = clean0["views_parse_status"].value_counts().to_dict()
print("parse status on the real file:", status_counts)
assert status_counts.get("missing", 0) == 0, "unexpected unparsed views"
assert status_counts.get("explicit_zero", 0) == 0, "unexpected explicit zeros"

,input,parsed
0,"'1,234'",1234.0
1,'12K',12000.0
2,'1.5M',1500000.0
3,'3.4万',34000.0
4,'1.2亿',120000000.0
5,'3350',3350.0
6,"' 19,000 '",19000.0
7,'12k次',12000.0
8,'',NaN
9,' ',NaN


parse status on the real file: {'parsed': 119}


### Distribution and the reason for a log transform

Views are extremely right-skewed (skew ≈ 8.4, kurtosis ≈ 78). One account has 3.34M views.
A raw mean would be dominated by that single row, so `log1p` is used for the attention
feature. The raw column is never overwritten.

In [8]:
vs = D.views_summary(clean0)
display(vs.set_index("statistic").T)
P.save_table(vs, "t01_views_summary")

statistic,n_non_missing,n_missing,n_explicit_zero,mean,std,min,p05,p25,median,p75,p95,max,skew_raw,skew_log,kurtosis_raw
value,119.0,0.0,0.0,85622.789916,334179.322731,3170.0,3624.5,7482.5,17000.0,49000.0,177000.0,3340000.0,8.378627,0.917442,78.338877


saved outputs/tables/t01_views_summary.csv


In [9]:
from matplotlib.ticker import FuncFormatter

fig, axes = P.plt.subplots(1, 2, figsize=(11, 3.8))

v = clean0["views_numeric"].astype(float)
axes[0].hist(v, bins=30, color=P.PALETTE["muted"], edgecolor="white")
axes[0].set_title("Views — raw scale (heavily right-skewed)")
axes[0].set_xlabel("views")
axes[0].set_ylabel("number of accounts")
axes[0].xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K"))

axes[1].hist(clean0["log_views"], bins=30, color=P.PALETTE["primary"], edgecolor="white")
axes[1].set_title("Views — log1p scale (approximately symmetric)")
axes[1].set_xlabel("log1p(views)")
axes[1].set_ylabel("number of accounts")

fig.suptitle("Figure 1. Views distribution before and after log transform (n=119 snapshot)",
             y=1.03, fontsize=12)
P.save_figure(fig, "fig01_views_distribution")

saved outputs/figures/fig01_views_distribution.png


## 4. Text: language, length and truncation

Language is a **deterministic script heuristic**, not a trained language identifier. It is good
enough to describe the corpus and to justify a multilingual encoder, and it is reported as an
approximation.

In [10]:
lang_counts = clean0["language"].value_counts()
display(lang_counts.to_frame("n_accounts").assign(pct=lambda d: (100 * d.n_accounts / len(clean0)).round(1)))

len_stats = clean0["text_length"].describe()
display(len_stats.to_frame("text_length (characters)"))

n_at_cap = int((clean0["text_length"] >= 250).sum())
print(f"posts at or beyond 250 characters: {n_at_cap} (max {int(clean0['text_length'].max())})")
print("interpretation: the upstream export truncates long posts around ~265 characters.")

,n_accounts,pct
language,,
zh,90,75.6
mixed,26,21.8
ja,2,1.7
en,1,0.8


,text_length (characters)
count,119.000000
mean,107.957983
std,49.966496
min,16.000000
25%,64.000000
50%,119.000000
75%,146.500000
max,263.000000


posts at or beyond 250 characters: 1 (max 263)
interpretation: the upstream export truncates long posts around ~265 characters.


In [11]:
fig, axes = P.plt.subplots(1, 2, figsize=(11, 3.8))

counts = lang_counts.sort_values(ascending=True)
axes[0].barh(counts.index, counts.values, color=P.PALETTE["primary"])
for y, x in enumerate(counts.values):
    axes[0].text(x + 1, y, str(x), va="center", fontsize=9)
axes[0].set_title("Detected script profile")
axes[0].set_xlabel("number of accounts")
axes[0].set_ylabel("language tag")

axes[1].hist(clean0["text_length"], bins=25, color=P.PALETTE["accent"], edgecolor="white")
axes[1].axvline(265, color="black", ls="--", lw=1)
axes[1].annotate("export truncation ≈265 chars", xy=(265, axes[1].get_ylim()[1]*0.85),
                 xytext=(150, axes[1].get_ylim()[1]*0.95), fontsize=8,
                 arrowprops=dict(arrowstyle="->", lw=0.8))
axes[1].set_title("Post length distribution")
axes[1].set_xlabel("characters after normalisation")
axes[1].set_ylabel("number of accounts")

fig.suptitle("Figure 2. Corpus composition (n=119 snapshot)", y=1.03, fontsize=12)
P.save_figure(fig, "fig02_language_and_length")

saved outputs/figures/fig02_language_and_length.png


## 5. Text normalisation and what it preserves

`normalise_text` applies NFKC normalisation, replaces URLs and @mentions with placeholders, and
collapses whitespace. It deliberately **keeps**:

- negation (`不`, `没有`, `别`);
- cashtags and hashtags;
- numbers and percentages;
- emoji and CJK punctuation-as-meaning.

No stemming or stop-word removal is applied. For Chinese text, stemming is meaningless and
removing short function words would destroy negation.

**URL note.** This export inserts spaces inside links (`https:// pan.quark.cn/...`), so a naive
`\S+` pattern finds nothing. The regex here reconstructs links across those inserted spaces.

In [12]:
examples = raw.loc[[0, 9, 47, 110], "text_raw"].tolist()
for t in examples:
    cleaned = D.normalise_text(t)
    print("RAW    :", t[:110].replace("\n", " / "), "...")
    print("CLEAN  :", cleaned[:110].replace("\n", " / "), "...")
    print("URLS   :", D.extract_urls(t))
    print("-" * 100)

print("rows containing at least one URL:", int(clean0["text_has_url"].sum()))
print("total URLs recovered             :", int(clean0["n_urls"].sum()))
print("cleaned text still containing 'http':", int(clean0["text_clean"].str.contains("http", case=False).sum()))

RAW    : AI中转站黑幕曝光：真钱买假模型已成行业潜规则！  很多低价Claude/GPT-5“满血”通道，其实偷偷换成国产弱模型/旧版3.5/甚至随机路由，赚差价跑路。论文都证实46%第三方API在掺假，学术引用都建立在假基石上 ...
CLEAN  : AI中转站黑幕曝光:真钱买假模型已成行业潜规则! 很多低价Claude/GPT-5“满血”通道,其实偷偷换成国产弱模型/旧版3.5/甚至随机路由,赚差价跑路.论文都证实46%第三方API在掺假,学术引用都建立在假基石上, ...
URLS   : []
----------------------------------------------------------------------------------------------------
RAW    : AI漫剧制作全流程，一口气学会从0到1制作首部作品  立即学习 https:// pan.quark.cn/s/ffb948fa81cf ...
CLEAN  : AI漫剧制作全流程,一口气学会从0到1制作首部作品 立即学习 <URL> ...
URLS   : ['https:// pan.quark.cn/s/ffb948fa81cf']
----------------------------------------------------------------------------------------------------
RAW    : RuView 利用WiFi透视墙壁。无需摄像头，无需可穿戴设备，无需互联网，只需无线电波。  https:// github.com/ruvnet/RuView ...
CLEAN  : RuView 利用WiFi透视墙壁.无需摄像头,无需可穿戴设备,无需互联网,只需无线电波. <URL> ...
URLS   : ['https:// github.com/ruvnet/RuView']
----------------------------------------------------------------------------------------------------
RAW    : 刚刚，吴恩达又发了新神器——Context Hub，拯救码农和agent们！ -

In [13]:
assert int((clean0["text_clean"].str.strip() == "").sum()) == 0, "blank cleaned text would be unmodellable"
assert clean0["text_length"].min() > 0
print("text integrity checks passed")

display(clean0.loc[0:2, ["row_id", "text_raw", "text_clean", "text_length", "n_urls", "n_hashtags", "n_cashtags"]])

text integrity checks passed


,row_id,text_raw,text_clean,text_length,n_urls,n_hashtags,n_cashtags
0,0,AI中转站黑幕曝光：真钱买假模型已成行业潜规则！ 很多低价Claude/GPT-5“满血”通道，其实偷偷换成国产弱模型/旧版3.5/甚至随机路由，赚差价跑路。论文都证实4...,"AI中转站黑幕曝光:真钱买假模型已成行业潜规则! 很多低价Claude/GPT-5“满血”通道,其实偷偷换成国产弱模型/旧版3.5/甚至随机路由,赚差价跑路.论文都证实46...",159,0,0,0
1,1,AI教父Yann LeCun说了一个事实。 一个四岁的小孩，看过的数据量，等于有史以来所有文本的总和。 最大的AI模型，训练数据是30万亿个单词。 听着很多？ 一个四岁小...,"AI教父Yann LeCun说了一个事实. 一个四岁的小孩,看过的数据量,等于有史以来所有文本的总和. 最大的AI模型,训练数据是30万亿个单词. 听着很多? 一个四岁小孩...",151,0,0,0
2,2,AI 用得越来越多以后，我发现了两个不一样的世界。 普通玩家：Prompt → Output（输入指令 → 得到结果） 高阶玩家：Context → Alignment...,"AI 用得越来越多以后,我发现了两个不一样的世界. 普通玩家:Prompt → Output(输入指令 → 得到结果) 高阶玩家:Context → Alignment →...",116,0,0,0


## 6. Near-duplicate detection

Exact duplicates and (author, text) duplicates are checked directly. Semantic near-duplicates
require embeddings, so they are computed here with the same compact multilingual encoder used
in notebook 02, then **flagged, not deleted**, as the specification requires.

If the encoder is unavailable in the environment, this section degrades gracefully and records
that near-duplicate detection was skipped — it never silently reports zero duplicates.

In [14]:
EMB_CACHE = ROOT / "outputs" / "tables" / "audit_embeddings.npz"
SIM_THRESHOLD = 0.92

near_dup_status = "not_run"
try:
    from src import features as F
    emb = F.embed_texts(clean0["text_clean"].tolist(),
                        cache_path=str(EMB_CACHE), batch_size=32)
    clean0 = F.attach_near_duplicate_flags(clean0, emb.vectors, threshold=SIM_THRESHOLD)
    sim = F.cosine_similarity_matrix(emb.vectors)
    off = sim[~np.eye(len(sim), dtype=bool)]
    near_dup_status = "ok"
    print("embedding model :", emb.model_name)
    print("device          :", emb.device)
    print("vectors         :", emb.vectors.shape)
    print("max off-diagonal cosine similarity:", round(float(off.max()), 4))
    print("pairs >= 0.92   :", int((np.triu(sim, 1) >= SIM_THRESHOLD).sum()))
    print("rows in a near-duplicate cluster:", int((clean0["near_dup_size"] > 1).sum()))
except Exception as exc:  # pragma: no cover - environment dependent
    near_dup_status = f"skipped: {type(exc).__name__}"
    print("near-duplicate detection skipped:", exc)

print("near_dup_status =", near_dup_status)

embedding model : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
device          : cuda
vectors         : (119, 384)
max off-diagonal cosine similarity: 0.7657
pairs >= 0.92   : 0
rows in a near-duplicate cluster: 0
near_dup_status = ok


In [15]:
if near_dup_status == "ok" and (clean0["near_dup_size"] > 1).any():
    clusters = clean0.loc[clean0["near_dup_size"] > 1, ["row_id", "nickname", "near_dup_cluster", "near_dup_size"]]
    display(clusters.sort_values(["near_dup_cluster", "row_id"]))
else:
    print("No near-duplicate clusters at cosine >= 0.92; nothing to display.")
    print("This is a genuine result for this snapshot, not a skipped check."
          if near_dup_status == "ok" else "Check was not executed.")

No near-duplicate clusters at cosine >= 0.92; nothing to display.
This is a genuine result for this snapshot, not a skipped check.


## 7. Investment-relevant entity mentions

A deliberately conservative keyword pass over the cleaned text, using the shared lexicon in
`src.features.ENTITY_LEXICON`. It is **not** the annotation layer: it only counts literal
surface mentions so the reader can see how rarely a specific listed company is named outright.
Matching is case-insensitive and includes both simplified and traditional Chinese forms, as the
corpus mixes scripts.


In [16]:
entity_df = F.count_entity_mentions(clean0["text_clean"].tolist())
P.save_table(entity_df, "t02_entity_mention_counts")
display(entity_df)

print("Tickers with zero literal mentions:",
      entity_df.loc[entity_df.n_posts_mentioning == 0, "ticker"].tolist())
print("Max mentions by a single ticker:",
      int(entity_df.n_posts_mentioning.max()), "posts",
      f"({entity_df.n_posts_mentioning.max() / len(clean0):.1%} of corpus)")


saved outputs/tables/t02_entity_mention_counts.csv


,ticker,aliases,n_posts_mentioning,pct_of_corpus
0,AAPL,"AAPL, Apple, 苹果, 蘋果",4,3.4
1,BABA,"BABA, Alibaba, 阿里巴巴, 阿里, 千问, 千問, Qwen",4,3.4
2,TENCENT,"Tencent, 腾讯, 騰訊",3,2.5
3,GOOGL,"GOOGL, Google, 谷歌",2,1.7
4,XIAOMI,"Xiaomi, 小米",2,1.7
5,BTC,"BTC, Bitcoin, 比特币, 比特幣",1,0.8
6,MU,"MU, Micron, 美光",1,0.8
7,PLTR,"PLTR, Palantir",1,0.8
8,TM,"Toyota, 丰田, 豐田",1,0.8
9,XPEV,"XPeng, 小鹏, 小鵬",1,0.8


Tickers with zero literal mentions: ['AMD', 'AMZN', 'AVGO', 'ETH', 'META', 'MSFT', 'NVDA', 'TSLA', 'TSM']
Max mentions by a single ticker: 4 posts (3.4% of corpus)


**Finding.** Even the most-mentioned ticker (AAPL, via "苹果") appears in a small minority of
posts, and most large-cap technology tickers never appear literally. Explicit single-name
coverage in this snapshot is too thin to build a cross-sectional equity signal. This motivates
the theme-level fallback used in the signal notebook, and it is one reason the theme bucket is
`NONE` for 98 of 119 posts.

## 8. Final data-quality summary

Everything a reviewer needs to judge the input, on one table.

In [17]:
dq = pd.DataFrame([
    {"check": "rows loaded", "value": len(clean0), "note": "matches supplied file"},
    {"check": "columns (canonical)", "value": 4, "note": "nickname, profile_url, views, latest post"},
    {"check": "missing values (raw fields)", "value": int(audit_missing.loc[audit_missing.column.isin(['nickname','profile_url','views_raw','text_raw']), 'n_missing'].sum()), "note": "none"},
    {"check": "exact duplicate rows", "value": int(raw.duplicated().sum()), "note": "none"},
    {"check": "exact duplicate cleaned texts", "value": int(clean0.is_duplicate_exact.sum()), "note": "none"},
    {"check": "duplicate nicknames", "value": int(raw.nickname.duplicated().sum()), "note": "same person, two handles; kept"},
    {"check": "views parsed successfully", "value": int((clean0.views_parse_status == 'parsed').sum()), "note": "all rows"},
    {"check": "views set to NaN by parser", "value": int(clean0.views_parse_status.eq('missing').sum()), "note": "invalid values are never coerced to 0"},
    {"check": "blank cleaned texts", "value": int((clean0.text_clean.str.strip() == '').sum()), "note": "none"},
    {"check": "posts with a URL", "value": int(clean0.text_has_url.sum()), "note": "URLs had inserted spaces upstream; recovered"},
    {"check": "post length, median (chars)", "value": int(clean0.text_length.median()), "note": f"median coincides with n=119 by chance; max {int(clean0.text_length.max())}, truncated by export"},
    {"check": "near-duplicate clusters (0.92)", "value": int((clean0.near_dup_size > 1).sum()), "note": near_dup_status},
    {"check": "timestamps available", "value": 0, "note": "blocks any historical backtest"},
    {"check": "post identifiers available", "value": 0, "note": "blocks de-duplication over time"},
    {"check": "price data supplied", "value": 0, "note": "blocks any return calculation"},
])
P.save_table(dq, "t03_data_quality_summary")
display(dq)

assert len(clean0) == 119
assert clean0["row_id"].is_unique
assert clean0["views_numeric"].notna().all()
print("\nAll data-quality assertions passed.")

saved outputs/tables/t03_data_quality_summary.csv


,check,value,note
0,rows loaded,119,matches supplied file
1,columns (canonical),4,"nickname, profile_url, views, latest post"
2,missing values (raw fields),0,none
3,exact duplicate rows,0,none
4,exact duplicate cleaned texts,0,none
5,duplicate nicknames,1,"same person, two handles; kept"
6,views parsed successfully,119,all rows
7,views set to NaN by parser,0,invalid values are never coerced to 0
8,blank cleaned texts,0,none
9,posts with a URL,7,URLs had inserted spaces upstream; recovered



All data-quality assertions passed.


## 9. Persist the processed snapshot

Raw and cleaned fields are written together. `posts_clean.csv` contains **no labels** — labels
are added in notebook 02 so that the annotation step stays visibly separate from the cleaning
step.

In [18]:
OUT = ROOT / "data" / "processed" / "posts_clean.csv"
clean0.to_csv(OUT, index=False, encoding="utf-8-sig")
print("written:", OUT.relative_to(ROOT), clean0.shape)
print("columns:", list(clean0.columns))

written: data/processed/posts_clean.csv (119, 21)
columns: ['row_id', 'nickname', 'profile_url', 'username', 'views_raw', 'views_numeric', 'views_parse_status', 'log_views', 'text_raw', 'text_clean', 'text_has_url', 'n_urls', 'n_cashtags', 'n_hashtags', 'language', 'text_length', 'word_count_latin', 'is_duplicate_exact', 'is_duplicate_author_text', 'near_dup_cluster', 'near_dup_size']


## 10. What this notebook establishes

1. The file is clean at the record level: 119 rows, no missing required values, no duplicate
   rows, one legitimately duplicated nickname.
2. Views are already numeric but are parsed defensively, and invalid values would become
   missing rather than zero.
3. The corpus is Chinese-dominant with heavy code-switching, short, and **truncated by the
   export**.
4. Explicit listed-company coverage is very thin.
5. There are **no timestamps, no post IDs and no prices**. A historical backtest is impossible
   with this input, and notebook 03 therefore specifies the design instead of inventing one.